In [ ]:
# src, configs
!cp -r /kaggle/input/czii-src /kaggle/working; mv /kaggle/working/czii-src /kaggle/working/src
!ln -s /kaggle/input/czii-configs /kaggle/working/configs

# for rootutils
!touch /kaggle/working/.project-root

In [ ]:
%%capture
!cd /kaggle/input/czii-pip-packages-v2; pip install --no-index --find-links=./packages -r requirements.txt

In [ ]:
experiment_list = [
    "250101-particle_hard_masks_r0.5-focalTverskyPp-pretrained_241221_299-hengck23_tf_efficientnetv2_b2_d64_256-s64_128-lr1e-3_decay05-bs4_2_2-ep100-transV3-preV4", # 0.766
    "250102-hard_r0.5-focalTverskyPp-pretrained_241205_299-monai_unet_d32_512_res1_head1_bn-s64_128-lr1e-3-bs4_2_2-ep100-transV1-preV1", # 0.763
    "250103-particle_hard_masks_r0.5-focalTverskyPp-hengck23_tf_efficientnetv2_b2_d64_256-s64_128-lr1e-3-bs4_2_2-ep100-transV3-preV4", # 0.753
    "250109-hard-focalTverskyPp-pretrained_241221_299-hengck23_v3_cn_nano_m2_d64_256-s64_128-lr1e-3_decay05-bs4_2_2-ep100-transV3-preV4", # 0.760
    "250111-focalTverskyPp-pretrained_241221_299-monai_segresnet_f16_bn_d1224-s64_128-lr1e-3_decay05-bs4_2_2-ep100-transV3-preV4", # 0.759
    "250113-focalTverskyPp-hengck23_enb2_d64_256-s64_256-lr1e-3-bs4_2_2-ep100-transV3-preV4", # 0.758
    "250116-focalTverskyPp-pretrained_241221_299-hengck23_resnet34d_d64_256-s64_256-lr1e-3-bs4_2_2-ep50-transV3-preV4", # 0.760
    "250117-focalTverskyPp-pretrained_241221_299-hengck23_resnet34d_d64_256-s64_256-lr1e-3-bs4_2_2-ep80-transV4-preV4", # 0.757
    "250118-focalTverskyPp-hengck23_env2b2_d64_256-s64_128-lr1e-3_decay05-bs4_2_2-mix_sim-ep100-transV4-preV4", # 0.757
    "250118-focalTverskyPp-hengck23_enb2_d64_256-disBA-s64_128-lr1e-3_decay05-bs4_2_2-ep100-transV4-preV4", # 0.757
]
compiled_model_dir_list = [
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
    "/kaggle/input/czii-sub-trt-models/czii_sub_trt_models",
]
fp16_mode_list = [
    "True",
    "False",
    "True",
    "True",
    "True",
    "True",
    "True",
    "True",
    "True",
    "True",
]
batch_size_pred_list = [
    "4",   
    "4",
    "4",
    "4",
    "4",   
    "4",
    "4",
    "4",
    "4",
    "4",   
]

copick_config_path = "/kaggle/input/czii-copick-config/copick_sub.config"
voxel_size = 10.012444
tomo_type = "denoised"

inference_overlap = ["0.25", "0.25", "0.25"]
discard_ratio = ["0.08", "0.08", "0.08"]
thresh = ["0.45", "0", "0.5", "0.6", "0.45", "0.7"]
n_tta = 1

ckpt_type = "last" # "last" or "best"

only_specific_particle = "None" # "None", 'apo-ferritin', 'beta-galactosidase', 'ribosome', 'thyroglobulin', 'virus-like-particle'

In [ ]:
# リストをスペース区切りの文字列に変換
experiment_list_str = " ".join(experiment_list)
compiled_model_dir_list_str = " ".join(compiled_model_dir_list)
fp16_mode_list_str = " ".join(fp16_mode_list)
batch_size_pred_list_str = " ".join(batch_size_pred_list)
thresh_str = " ".join(thresh)
inference_overlap_str = " ".join(inference_overlap)
discard_ratio_str = " ".join(discard_ratio)

# コマンドを組み立て
cmd = (
    "cp /kaggle/input/czii-scripts/submit_v2_1_all_data.py /kaggle/working/src; "
    "python /kaggle/working/src/submit_v2_1_all_data.py "
    f"--copick_config_path {copick_config_path} "
    f"--experiment_list {experiment_list_str} "
    f"--compiled_model_dir_list {compiled_model_dir_list_str} "
    f"--fp16_mode_list {fp16_mode_list_str} "
    f"--voxel_size {voxel_size} "
    f"--tomo_type {tomo_type} "
    f"--inference_overlap {inference_overlap_str} "
    f"--discard_ratio {discard_ratio_str} "
    f"--thresh {thresh_str} "
    f"--batch_size_pred_list {batch_size_pred_list_str} "
    f"--n_tta {n_tta} "
    f"--ckpt_type {ckpt_type} "
    f"--only_specific_particle {only_specific_particle}"
) 

# コマンドを実行
!{cmd}

In [ ]:
from glob import glob
import matplotlib.pyplot as plt
import torch


z_list = [40, 80, 120]
C = 6

pred_path = glob("/kaggle/tmp/preds/preds_fold_mean*.pt")
if len(pred_path) > 0:
    preds = torch.load(pred_path[0])
    preds = preds.cpu().numpy()

    plt.figure(figsize=(18, 10))
    for i, z in enumerate(z_list):
        for c in range(C):
            plt.subplot(len(z_list), C, i * C + c + 1)
            plt.imshow(preds[c, z], cmap="viridis", vmin=0, vmax=1)
            plt.axis("off")